# Run Evaluation (Python port)

Python port of the OpenML run-evaluation pipeline. Mirrors these Java sources:

- `EvaluateRun.java` — top-level dispatcher
- `EvaluateBatchPredictions.java` — folds + samples evaluator
- `EvaluateStreamPredictions.java` — lockstep stream evaluator
- `EvaluateSurvivalAnalysisPredictions.java` — count-validation-only stub
- `OpenMLEvaluation.java` + `Output.java` — metric formulas
- `InstancesHelper.java` — `toProbDist`, `predictionToConfidences`, etc.
- `FoldsPredictionCounter.java` — TEST/TRAIN row-count validation

Out of scope (will land in a later phase): the upload pipeline (`RunEvaluation`
POST, `RunTrace`), trace parsing, run-description parsing, and the consistency
check against user-defined measures. All examples here assume the dataset,
splits, and predictions are already loaded in memory.


In [ ]:
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from process_dataset import load_dataset

from models import EstimationProcedure, EstimationProcedureType
from process_dataset import generate_folds
from runs import evaluate_run, evaluate_survival, evaluate_batch, evaluate_stream, regression_metrics, classification_metrics, TaskType, FoldsPredictionCounter

# The tiny synthetic tasks below legitimately trigger UndefinedMetric warnings
# (e.g. single-class folds); silence them so the notebook reads cleanly.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

print("imports OK")

## Example 1 — Regression on synthetic data

Hand-verify MAE/RMSE on a 4-row toy regression problem. With
`y_train = [1, 2, 3, 4]` the prior is `mean = 2.5`; with
`y_pred = [1.5, 2.5, 2.5, 4.5]` the per-instance absolute errors are all `0.5`,
so MAE = RMSE = 0.5. The "prior" model predicts `mean(y_train) = 2.5` for every
instance; its MAE is `(1.5 + 0.5 + 0.5 + 1.5) / 4 = 1.0`, so
`relative_absolute_error = 0.5 / 1.0 = 0.5`.


In [ ]:
y_true = np.array([1.0, 2.0, 3.0, 4.0])
y_pred = np.array([1.5, 2.5, 2.5, 4.5])
y_train = np.array([1.0, 2.0, 3.0, 4.0])

m = regression_metrics(y_true, y_pred, y_train)
for k, v in m.items():
    print(f"  {k:35s} = {v:.6f}")

assert abs(m["mean_absolute_error"] - 0.5) < 1e-12
assert abs(m["root_mean_squared_error"] - 0.5) < 1e-12
assert abs(m["mean_prior_absolute_error"] - 1.0) < 1e-12
assert abs(m["relative_absolute_error"] - 0.5) < 1e-12
print("\nhand-computed values match ✓")


## Example 2 — Binary classification with confidences

A 4-instance / 2-class problem.

| row | true | pred | conf.neg | conf.pos |
|-----|------|------|----------|----------|
| 0   | neg  | neg  | 0.8      | 0.2      |
| 1   | neg  | pos  | 0.4      | 0.6      |
| 2   | pos  | pos  | 0.3      | 0.7      |
| 3   | pos  | pos  | 0.2      | 0.8      |

Confusion matrix should be `[[1, 1], [0, 2]]`, accuracy 3/4 = 0.75, and
unweighted recall = (0.5 + 1.0) / 2 = 0.75 (mean of per-class recalls, not
weighted by class frequency).


In [ ]:
class_names = ["neg", "pos"]
y_true = [0, 0, 1, 1]
y_pred = [0, 1, 1, 1]
conf = np.array([
    [0.8, 0.2],
    [0.4, 0.6],
    [0.3, 0.7],
    [0.2, 0.8],
])
y_train = [0, 0, 1, 1]

cm = classification_metrics(y_true, y_pred, conf, class_names, y_train)

print(f"  predictive_accuracy  : {cm['predictive_accuracy']:.4f}")
print(f"  kappa                : {cm['kappa']:.4f}")
print(f"  area_under_roc_curve : {cm.get('area_under_roc_curve')}")
print(f"  unweighted_recall    : {cm['unweighted_recall']:.4f}")
print(f"  weighted_recall      : {cm['weighted_recall']:.4f}")
print(f"  f_measure            : {cm['f_measure']:.4f}")
print(f"  prior_entropy        : {cm['prior_entropy']:.4f}")
print(f"  confusion_matrix     : {cm['confusion_matrix']}")
print(f"  per-class precision  : {cm['_per_class']['precision']}")
print(f"  per-class recall     : {cm['_per_class']['recall']}")
print(f"  per-class auroc      : {cm['_per_class']['auroc']}")

assert abs(cm["predictive_accuracy"] - accuracy_score(y_true, y_pred)) < 1e-12
assert cm["confusion_matrix"] == [[1, 1], [0, 2]]
assert abs(cm["unweighted_recall"] - 0.75) < 1e-12
print("\ncross-checks pass ✓")


## Example 3 — Multiclass iris, 5-fold CV end-to-end

Realistic batch evaluation. We:

1. Download iris (did=61) via the existing `load_dataset` helper.
2. Generate 5-fold stratified CV splits via `generate_folds`.
3. For each `(repeat, fold)`, train a `LogisticRegression` on the TRAIN rows
   and produce `predict_proba` over the TEST rows.
4. Assemble a predictions DataFrame with
   `row_id, repeat, fold, prediction, confidence.<class>` columns.
5. Run `evaluate_batch` and confirm the global `predictive_accuracy` matches
   `sklearn.metrics.accuracy_score` over the concatenated test predictions.

First run needs network to download iris; the temp file is cached afterward
by `src.helpers.get_data_and_meta_information_from_did`.


In [ ]:
df, target = load_dataset(61)
class_names = df[target].cat.categories.tolist()
print(f"iris: {len(df)} rows, target = {target!r}, classes = {class_names}")

cv_splits, _, _ = generate_folds(
    did=61,
    procedure=EstimationProcedure(
        type=EstimationProcedureType.CROSSVALIDATION,
        folds=5,
        repeats=1,
    ),
    seed=1,
)
print(
    f"splits: {len(cv_splits)} rows, "
    f"TRAIN={len(cv_splits[cv_splits.type == 'TRAIN'])}, "
    f"TEST={len(cv_splits[cv_splits.type == 'TEST'])}"
)

# Encode features/target to numeric for sklearn.
feature_cols = [c for c in df.columns if c != target]
X = df[feature_cols].apply(lambda c: pd.Categorical(c).codes if c.dtype == object else c).to_numpy(dtype=float)
y = df[target].cat.codes.to_numpy()

pred_rows = []
test_mask = cv_splits.type == "TEST"
for (repeat, fold), grp in cv_splits[test_mask].groupby(["repeat", "fold"]):
    test_rowids = grp["rowid"].to_numpy()
    train_mask = (
        (cv_splits["repeat"] == repeat)
        & (cv_splits["fold"] == fold)
        & (cv_splits["type"] == "TRAIN")
    )
    train_rowids = cv_splits.loc[train_mask, "rowid"].to_numpy()

    clf = LogisticRegression(max_iter=200)
    clf.fit(X[train_rowids], y[train_rowids])
    proba = clf.predict_proba(X[test_rowids])
    pred_idx = np.argmax(proba, axis=1)

    for i, rid in enumerate(test_rowids):
        row = {
            "row_id": int(rid),
            "repeat": int(repeat),
            "fold": int(fold),
            "prediction": class_names[int(pred_idx[i])],
        }
        for j, cls in enumerate(class_names):
            row[f"confidence.{cls}"] = float(proba[i, j])
        pred_rows.append(row)

predictions_df = pd.DataFrame(pred_rows)
print(f"predictions: {len(predictions_df)} rows")
print(predictions_df.head())

per_cell, global_scores, counter = evaluate_batch(
    df,
    cv_splits,
    predictions_df,
    target,
    TaskType.CLASSIFICATION,
    estimation_procedure_type=EstimationProcedureType.CROSSVALIDATION,
)
print(f"\nFoldsPredictionCounter.check(): {counter.check()}")
print(f"per_cell scores: {len(per_cell)}")
print(f"global scores:   {len(global_scores)}")

by_func = {s.function: s for s in global_scores}
acc = by_func["predictive_accuracy"].value
acc_stdev = by_func["predictive_accuracy"].stdev
print(f"\npredictive_accuracy  : {acc:.4f} ± {acc_stdev:.4f}")
print(f"kappa                : {by_func['kappa'].value:.4f}")
print(f"f_measure            : {by_func['f_measure'].value:.4f}")
print(f"area_under_roc_curve : {by_func.get('area_under_roc_curve', None)}")

# Cross-check against sklearn accuracy on the concatenated test predictions.
y_true_concat = y[predictions_df["row_id"].to_numpy()]
y_pred_concat = np.array([class_names.index(p) for p in predictions_df["prediction"]])
sklearn_acc = accuracy_score(y_true_concat, y_pred_concat)
print(f"\nsklearn accuracy (concatenated): {sklearn_acc:.4f}")
assert abs(acc - sklearn_acc) < 1e-9
print("matches sklearn ✓")


## Example 4 — Stream evaluation (task_type_id = 4)

Stream tasks evaluate predictions in lockstep with the dataset — every
`row_id` must equal the instance index. We synthesize a 4-row stream,
compute the metrics, then flip a `row_id` to demonstrate the ordering check.


In [ ]:
stream_dataset = pd.DataFrame({
    "x": [1.0, 2.0, 3.0, 4.0],
    "label": pd.Categorical(["a", "a", "b", "b"], categories=["a", "b"]),
})
stream_preds = pd.DataFrame([
    {"row_id": 0, "prediction": "a", "confidence.a": 0.9, "confidence.b": 0.1},
    {"row_id": 1, "prediction": "a", "confidence.a": 0.7, "confidence.b": 0.3},
    {"row_id": 2, "prediction": "b", "confidence.a": 0.2, "confidence.b": 0.8},
    {"row_id": 3, "prediction": "b", "confidence.a": 0.1, "confidence.b": 0.9},
])

stream_scores = evaluate_stream(stream_dataset, stream_preds, "label")
for s in stream_scores:
    if s.value is not None:
        print(f"  {s.function:35s} = {s.value:.4f}")

# Out-of-order row_id should raise.
bad_preds = stream_preds.copy()
bad_preds.loc[0, "row_id"] = 2  # collides with row 2
try:
    evaluate_stream(stream_dataset, bad_preds, "label")
    print("\nERROR: expected ValueError")
except ValueError as e:
    print(f"\nexpected error: {e}")


## Example 5 — Survival analysis (task_type_id = 7, count validation only)

The Java `EvaluateSurvivalAnalysisPredictions` only validates prediction
counts and bounds; `evaluationScores` is left empty
(see EvaluateSurvivalAnalysisPredictions.java:89-91). This port reproduces
that behavior faithfully — a placeholder until `sksurv` metrics are wired in.


In [ ]:
surv_dataset = pd.DataFrame({"time": [10.0, 20.0, 30.0, 40.0]})
surv_splits = pd.DataFrame([
    ("TRAIN", 0, 0, 0), ("TRAIN", 1, 0, 0),
    ("TEST",  2, 0, 0), ("TEST",  3, 0, 0),
    ("TEST",  0, 0, 1), ("TEST",  1, 0, 1),
    ("TRAIN", 2, 0, 1), ("TRAIN", 3, 0, 1),
], columns=["type", "rowid", "repeat", "fold"])
surv_preds = pd.DataFrame([
    {"row_id": 2, "repeat": 0, "fold": 0},
    {"row_id": 3, "repeat": 0, "fold": 0},
    {"row_id": 0, "repeat": 0, "fold": 1},
    {"row_id": 1, "repeat": 0, "fold": 1},
])

scores, pc = evaluate_survival(surv_dataset, surv_splits, surv_preds)
print(f"scores: {scores}  (empty, as in Java)")
print(f"prediction_counter.check(): {pc.check()}")
print(f"expected_total: {pc.get_expected_total()}")

# Drop one prediction to break the count.
bad_preds = surv_preds.drop(index=0)
try:
    evaluate_survival(surv_dataset, surv_splits, bad_preds)
    print("\nERROR: expected RuntimeError")
except RuntimeError as e:
    print(f"\nexpected error: {e}")


## Top-level dispatcher

`evaluate_run(task_type_id, ...)` routes to the right evaluator and packages
the result as a `RunEvaluation`. Unsupported task types and missing
predictions short-circuit with `error` set rather than raising — same pattern
the Java code uses at `EvaluateRun.java:102-134`.


In [ ]:
result = evaluate_run(
    task_type_id=1,
    dataset_df=df,
    splits_df=cv_splits,
    predictions_df=predictions_df,
    target_feature=target,
    estimation_procedure_type=EstimationProcedureType.CROSSVALIDATION,
    run_id=42,
)
print(f"run_id: {result.run_id}")
print(f"error : {result.error}")
print(f"# scores: {len(result.scores)}")

# Unsupported task type short-circuits with an error.
bad = evaluate_run(99, df, cv_splits, predictions_df, target)
print(f"\nunsupported task_type_id=99 -> error: {bad.error!r}")
